# LAB 5 — LangFuse: Visualização de Custo e Traces
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Instrumentar os pipelines Naive RAG, Multi-Query RAG e RAG-Fusion com LangFuse para comparar visualmente o custo real (tokens + latência) de cada abordagem. Calcular o custo por 1.000 queries de cada técnica.

**Tempo estimado:** 20 minutos

---
**Checklist de entrega:**
- [ ] Traces das 3 abordagens visíveis no LangFuse
- [ ] Cálculo de custo por 1.000 queries documentado
- [ ] Conclusão escrita sobre qual abordagem usar

In [ ]:
!pip install -q langchain langchain-openai sentence-transformers opensearch-py langfuse pandas matplotlib
print('✅ Dependências instaladas!')

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch

# Configurações LangFuse
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-sua-chave')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-sua-chave')
LANGFUSE_HOST = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')

# Inicializar LangFuse
langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST
)

# Outros clientes
llm = ChatOpenAI(
    base_url=os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1'),
    api_key='dummy',
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.3
)
embeddings = SentenceTransformer('BAAI/bge-m3')
os_client = OpenSearch(
    hosts=[{'host': os.getenv('OPENSEARCH_HOST', 'localhost'), 'port': 9200}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False, verify_certs=False
)

INDEX_NAME = 'corpus_juridico_aula7'
print('✅ Ambiente configurado!')
print(f'LangFuse: {LANGFUSE_HOST}')

## 2. Pipelines Instrumentados com LangFuse

In [ ]:
def retrieve_knn(query: str, k: int = 5) -> list:
    """Retrieval kNN básico."""
    q_vec = embeddings.encode(query, normalize_embeddings=True).tolist()
    resp = os_client.search(
        index=INDEX_NAME,
        body={'size': k, 'query': {'knn': {'embedding': {'vector': q_vec, 'k': k}}},
              '_source': ['content']}
    )
    return [{'id': h['_id'], 'content': h['_source'].get('content', '')} 
            for h in resp['hits']['hits']]

# ── Pipeline 1: Naive RAG ─────────────────────────────────────────────
@observe(name='naive_rag')
def naive_rag_pipeline(query: str) -> dict:
    """Pipeline Naive RAG com trace LangFuse."""
    t0 = time.time()
    
    with langfuse_context.observe_span('retrieval'):
        docs = retrieve_knn(query, k=5)
    
    contexto = '\n'.join(d['content'][:200] for d in docs)
    
    with langfuse_context.observe_generation('llm_generation',
                                              model='llama-3.1-8b-instruct',
                                              input=query):
        resp = llm.invoke(f'Contexto:\n{contexto}\n\nPergunta: {query}\n\nResposta:')
        tokens_estimados = len(resp.content.split()) * 2  # estimativa
        langfuse_context.update_current_observation(output=resp.content,
                                                     usage={'total_tokens': tokens_estimados})
    
    return {
        'resposta': resp.content,
        'latencia': time.time() - t0,
        'tokens': tokens_estimados,
        'n_docs': len(docs)
    }

# ── Pipeline 2: Multi-Query RAG (N=3) ────────────────────────────────
@observe(name='multi_query_rag_n3')
def multi_query_rag_pipeline(query: str, n: int = 3) -> dict:
    """Pipeline Multi-Query RAG N=3 com trace LangFuse."""
    t0 = time.time()
    todos_docs = {}
    tokens_gen = 0
    
    PROMPT = f"""Gere {n-1} variações da pergunta. Retorne uma por linha:
{query}
Variações:"""
    
    with langfuse_context.observe_generation('query_expansion',
                                              model='llama-3.1-8b-instruct',
                                              input=query):
        resp_var = llm.invoke(PROMPT)
        variacoes = [v.strip() for v in resp_var.content.strip().split('\n') if v.strip()]
        all_queries = [query] + variacoes[:n-1]
        tokens_gen = len(PROMPT.split()) + len(resp_var.content.split())
        langfuse_context.update_current_observation(output=str(all_queries),
                                                     usage={'total_tokens': tokens_gen})
    
    with langfuse_context.observe_span('multi_retrieval'):
        for q in all_queries:
            for doc in retrieve_knn(q, k=5):
                todos_docs[doc['id']] = doc
    
    contexto = '\n'.join(d['content'][:150] for d in list(todos_docs.values())[:8])
    
    with langfuse_context.observe_generation('llm_generation',
                                              model='llama-3.1-8b-instruct',
                                              input=query):
        resp = llm.invoke(f'Contexto:\n{contexto}\n\nPergunta: {query}\n\nResposta:')
        tokens_gen2 = len(resp.content.split()) * 2
        langfuse_context.update_current_observation(output=resp.content,
                                                     usage={'total_tokens': tokens_gen2})
    
    return {
        'resposta': resp.content,
        'latencia': time.time() - t0,
        'tokens': tokens_gen + tokens_gen2,
        'n_docs': len(todos_docs)
    }

print('✅ Pipelines instrumentados com LangFuse!')

## 3. Executar os 3 Pipelines e Coletar Métricas

In [ ]:
queries_teste = [
    'Quais são os requisitos para prisão preventiva?',
    'Como funciona a busca e apreensão domiciliar?',
    'Quais são os direitos do preso em flagrante?'
]

metricas = {'Naive RAG': [], 'Multi-Query N=3': []}

for query in queries_teste:
    print(f'\nProcessando: {query[:50]}...')
    
    # Naive RAG
    r1 = naive_rag_pipeline(query)
    metricas['Naive RAG'].append(r1)
    print(f'  Naive RAG: {r1["latencia"]:.2f}s | {r1["tokens"]} tokens | {r1["n_docs"]} docs')
    
    # Multi-Query N=3
    r2 = multi_query_rag_pipeline(query, n=3)
    metricas['Multi-Query N=3'].append(r2)
    print(f'  Multi-Query N=3: {r2["latencia"]:.2f}s | {r2["tokens"]} tokens | {r2["n_docs"]} docs')

# Flush LangFuse
langfuse.flush()
print(f'\n✅ Traces enviados ao LangFuse: {LANGFUSE_HOST}')

## 4. Cálculo de Custo por 1.000 Queries

In [ ]:
import numpy as np

PRECO_POR_1M_TOKENS_INPUT = 0.15   # USD (ex: GPT-4o-mini)
PRECO_POR_1M_TOKENS_OUTPUT = 0.60  # USD
VOLUME_QUERIES = 1000

print('=== ANÁLISE DE CUSTO POR 1.000 QUERIES ===')
print()
print(f'Modelo de referência (API externa): GPT-4o-mini equivalente')
print(f'Preço: ${PRECO_POR_1M_TOKENS_INPUT}/1M tokens input, ${PRECO_POR_1M_TOKENS_OUTPUT}/1M tokens output')
print()

tabela_custo = []

for abordagem, resultados in metricas.items():
    tokens_medios = np.mean([r['tokens'] for r in resultados])
    lat_media = np.mean([r['latencia'] for r in resultados])
    
    # Dividir tokens: ~60% input, ~40% output (estimativa)
    tokens_input = tokens_medios * 0.6
    tokens_output = tokens_medios * 0.4
    
    custo_por_query = (
        (tokens_input / 1_000_000) * PRECO_POR_1M_TOKENS_INPUT +
        (tokens_output / 1_000_000) * PRECO_POR_1M_TOKENS_OUTPUT
    )
    custo_1000 = custo_por_query * VOLUME_QUERIES
    
    print(f'{abordagem}:')
    print(f'  Tokens médios por query: {tokens_medios:.0f}')
    print(f'  Latência média: {lat_media:.2f}s')
    print(f'  Custo por query: ${custo_por_query:.5f}')
    print(f'  Custo por 1.000 queries: ${custo_1000:.2f}')
    print()
    
    tabela_custo.append({
        'Abordagem': abordagem,
        'Tokens/query': int(tokens_medios),
        'Latência (s)': round(lat_media, 2),
        'Custo/query ($)': round(custo_por_query, 5),
        'Custo/1k queries ($)': round(custo_1000, 2)
    })

df_custo = pd.DataFrame(tabela_custo)
print('=== TABELA RESUMO ===')
print(df_custo.to_string(index=False))
print()
print('💡 NOTA: Com vLLM local (Llama 3.1 8B), o custo monetário é ~$0')
print('   O custo real é computacional: GPU time × preço de energia')

## 5. Visualização Comparativa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Comparação de Custo e Latência: Naive RAG vs Multi-Query\n(3 queries jurídicas)', fontsize=12)

abordagens = df_custo['Abordagem'].values
x = range(len(abordagens))

# Gráfico 1: Latência
axes[0].bar(x, df_custo['Latência (s)'], color=['#2ecc71', '#e74c3c'], alpha=0.8)
axes[0].set_title('Latência Média por Query', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(abordagens, rotation=10)
axes[0].set_ylabel('Segundos')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_custo['Latência (s)']):
    axes[0].text(i, v + 0.05, f'{v:.2f}s', ha='center', va='bottom')

# Gráfico 2: Custo por 1000 queries
axes[1].bar(x, df_custo['Custo/1k queries ($)'], color=['#3498db', '#9b59b6'], alpha=0.8)
axes[1].set_title('Custo por 1.000 Queries\n(API externa estimada)', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(abordagens, rotation=10)
axes[1].set_ylabel('USD ($)')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_custo['Custo/1k queries ($)']):
    axes[1].text(i, v + 0.01, f'${v:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('langfuse_custo_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo como langfuse_custo_comparativo.png')

## 6. Conclusão

Escreva sua análise final abaixo:

### Qual abordagem usar? Por quê?

Com base nos traces do LangFuse e nas métricas de custo:

1. **Para um chatbot público** (alto volume, SLA de latência < 2s): [sua análise]

2. **Para uma ferramenta interna de investigação** (baixo volume, alta precisão): [sua análise]

3. **Fator surpresa** (algo que você não esperava dos resultados): [sua análise]

> **Complete sua análise aqui**

In [ ]:
print('=== CHECKLIST DE ENTREGA — LAB 5 ===')
checklist = [
    'Traces das abordagens enviados e visíveis no LangFuse',
    'Cálculo de custo por 1.000 queries documentado (com fórmula)',
    'Visualização comparativa gerada',
    'Conclusão escrita sobre qual abordagem usar'
]
for item in checklist:
    print(f'  [ ] {item}')
print('\n✅ Lab 5 concluído! Prossiga para o LAB 6 — Langflow (bônus).')